# Removendo Tendência Determinística do IPCA com Python

### Análise de séries temporais com dados do IBGE/SIDRA

Este notebook analisa o **número-índice mensal do IPCA** entre janeiro de 2000 e abril de 2019. O objetivo é aplicar, em Python, dois procedimentos apresentados no estudo de tendências em séries temporais:

- estimação e remoção de uma **tendência determinística linear**;
- transformação da série por **primeira diferença**.

Como a tabela utilizada no exercício original não foi disponibilizada, os dados foram obtidos diretamente da **Tabela 1737 do SIDRA/IBGE**. Para reproduzir a proposta do exercício, o número-índice foi rebaseado para **dezembro de 1999 = 100**.


## 1. Bibliotecas


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression


## 2. Obtenção e preparação dos dados

A série é consultada diretamente na API do SIDRA/IBGE para o período de janeiro de 2000 a abril de 2019.


In [ ]:
url = "https://apisidra.ibge.gov.br/values/t/1737/n1/all/v/2266/p/200001-201904?formato=json"
dados_ibge = pd.read_json(url)

dados = dados_ibge.iloc[1:].copy()
dados = dados[["D3N", "V"]]
dados.columns = ["data", "IPCA_original"]
dados["IPCA_original"] = pd.to_numeric(dados["IPCA_original"])

# Índice temporal: t = 1, 2, ..., 232
dados["mes"] = np.arange(1, len(dados) + 1)

dados.head()


## 3. Mudança de base

O exercício utiliza **dezembro de 1999 = 100**. Na série consultada no SIDRA, o número-índice de dezembro de 1999 é 1588,56.

A mudança de base é realizada por:

$$
I_t^{(nova\ base)} =
\frac{I_t}{I_{dez/1999}}\times 100
$$


In [ ]:
url_base = "https://apisidra.ibge.gov.br/values/t/1737/n1/all/v/2266/p/199912?formato=json"
base_ibge = pd.read_json(url_base)

base_dez_1999 = float(base_ibge.loc[1, "V"])
dados["IPCA"] = (dados["IPCA_original"] / base_dez_1999) * 100

print(f"Base dez/1999: {base_dez_1999:.2f}")
print(f"Número de observações: {len(dados)}")
dados.head()


## 4. Série temporal original

A visualização da série permite identificar uma trajetória crescente do número-índice ao longo do período, sugerindo a presença de tendência.


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(dados["mes"], dados["IPCA"])
plt.title("IPCA — janeiro/2000 a abril/2019")
plt.xlabel("Tempo (meses)")
plt.ylabel("Número-índice do IPCA")
plt.grid(True, alpha=0.3)
plt.show()


## 5. Estimação da tendência determinística

Uma tendência linear pode ser representada por:

$$
\hat{\tau}_t = \beta_0 + \beta_1t
$$

em que $\beta_0$ é o intercepto, $\beta_1$ é a inclinação da reta e $t$ representa o tempo.


In [ ]:
X = dados[["mes"]]
y = dados["IPCA"]

modelo = LinearRegression()
modelo.fit(X, y)

beta0 = modelo.intercept_
beta1 = modelo.coef_[0]

print(f"β0 (intercepto): {beta0:.4f}")
print(f"β1 (inclinação): {beta1:.4f}")
print(f"Tendência estimada: τ̂t = {beta0:.4f} + {beta1:.4f}t")


Com os dados utilizados neste notebook, a tendência estimada é aproximadamente:

$$
\hat{\tau}_t = 86{,}7976 + 0{,}9737t
$$

O coeficiente positivo de $\beta_1$ indica crescimento do **número-índice** ao longo do tempo. Ele não representa uma taxa mensal de inflação.


In [ ]:
dados["tendencia"] = modelo.predict(X)

plt.figure(figsize=(12, 5))
plt.plot(dados["mes"], dados["IPCA"], label="IPCA observado")
plt.plot(dados["mes"], dados["tendencia"], label="Tendência linear")
plt.title("IPCA e tendência determinística linear")
plt.xlabel("Tempo (meses)")
plt.ylabel("Número-índice do IPCA")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 6. Remoção da tendência

A série sem a tendência linear é obtida por:

$$
Y_t - \hat{\tau}_t
$$

A diferença entre o valor observado e o valor estimado pela regressão corresponde ao **resíduo**.


In [ ]:
dados["sem_tendencia"] = dados["IPCA"] - dados["tendencia"]

dados[["data", "IPCA", "tendencia", "sem_tendencia"]].head()


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(dados["mes"], dados["sem_tendencia"], label="Série sem tendência")
plt.axhline(0, linestyle="--")
plt.title("IPCA após a remoção da tendência determinística")
plt.xlabel("Tempo (meses)")
plt.ylabel("Resíduos")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


### Interpretação

Após a remoção da tendência linear, desaparece a forte trajetória crescente observada na série original. Entretanto, os resíduos ainda apresentam uma estrutura visível ao longo do tempo.

Esse resultado é relevante porque mostra que uma reta é uma simplificação: embora ela represente a direção geral de crescimento, não captura integralmente a dinâmica da série. A transformação dos dados também evidencia como diferentes representações da mesma série podem revelar aspectos distintos do comportamento temporal.


## 7. Primeira diferença

Outro procedimento é calcular a mudança entre observações consecutivas:

$$
\Delta Y_t = Y_t - Y_{t-1}
$$

Nesse caso, não se estima previamente uma reta. A transformação mostra quanto o número-índice mudou em relação ao período imediatamente anterior.


In [ ]:
dados["diferenca"] = dados["IPCA"].diff()

dados[["data", "IPCA", "diferenca"]].head()


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(dados["mes"], dados["diferenca"], label="Primeira diferença")
plt.axhline(0, linestyle="--")
plt.title("Primeira diferença do IPCA")
plt.xlabel("Tempo (meses)")
plt.ylabel("Diferença em pontos do número-índice")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


A primeira observação resulta em `NaN`, pois não existe uma observação anterior dentro do período analisado.

A primeira diferença representa uma **diferença em pontos do número-índice**. Portanto, ela não deve ser interpretada diretamente como a taxa percentual mensal do IPCA.


## 8. Comparação dos métodos

A análise mostrou duas formas distintas de trabalhar com a tendência:

**Regressão linear:** estima uma trajetória determinística ao longo do tempo e subtrai essa trajetória da série. O resultado são os resíduos.

**Primeira diferença:** calcula diretamente a mudança entre dois períodos consecutivos.

Os gráficos deixam evidente que uma transformação pode alterar profundamente a forma visual de uma série sem alterar os dados de origem: cada representação responde a uma pergunta diferente sobre o comportamento temporal.


## 9. Observação sobre o exercício original

O material de referência apresenta:

$$
\beta_0 = 84{,}7007
$$

$$
\beta_1 = 0{,}9891
$$

Neste notebook foram obtidos aproximadamente:

$$
\beta_0 = 86{,}7976
$$

$$
\beta_1 = 0{,}9737
$$

Os resultados são próximos, mas não idênticos. Como a tabela original utilizada no exercício não foi fornecida, a série deste projeto foi reconstruída a partir dos dados do SIDRA/IBGE e rebaseada para dezembro de 1999 = 100.

Por esse motivo, foram preservados os coeficientes efetivamente obtidos com os dados utilizados, em vez de ajustar os resultados para coincidir artificialmente com o material de referência.


## 10. Conclusão

A série original do número-índice do IPCA apresenta uma trajetória crescente clara. A regressão linear permitiu estimar e remover uma tendência determinística, enquanto a primeira diferença evidenciou as mudanças entre períodos consecutivos.

A comparação entre os gráficos mostra um aspecto central da análise de séries temporais: **os mesmos dados podem revelar estruturas muito diferentes dependendo da transformação aplicada**. No caso analisado, a permanência de um padrão nos resíduos também indica que uma tendência linear simples não descreve completamente o comportamento temporal da série.

### Tecnologias

`Python` · `Pandas` · `NumPy` · `Matplotlib` · `Scikit-learn` · `Google Colab`

### Fonte

**IBGE/SIDRA — Tabela 1737**  
Período analisado: janeiro de 2000 a abril de 2019.
